# N21 — FINAL N07 two-ViT ensemble FER replay on N17v2 (12 × 2)

**Run all in Google Colab (CPU or GPU).** No diffusion or biometric generation. Strict frozen N07 two-model softmax-probability mean, human-target emotion preservation, N07 12-original replay gate, SHA-256 verification, source-cluster bootstrap. Privacy-sensitive row-level data remain in private Google Drive. The run **fails closed** when original N07 predictions do not match. Do not substitute single-ViT N20 scores. This notebook produces only provisional scientific claims until manual source-face QC is completed.

In [ ]:
%pip -q install 'transformers==4.44.2' 'huggingface-hub==0.36.2' 'pandas>=2.2,<3' 'Pillow>=10,<12' 'torch>=2.2' 'numpy>=1.26,<3'
from google.colab import drive
drive.mount('/content/drive')
import hashlib, json
from pathlib import Path
import numpy as np, pandas as pd, torch
from PIL import Image
from transformers import AutoImageProcessor, AutoModelForImageClassification
R=Path('/content/drive/MyDrive/thesis_all/outputs/runs')
N07=R/'20260818T210720Z_notebook07_affectnet_larger_6aeb3643_f1142664'
N17=R/'nb17v2_neutral_expression_12sources_2seeds_sd3pinned'
N18=R/'nb18_paired_neutral_pilot_analysis_v1'
N19=R/'nb19_n17v2_frozen_v3_biometrics'
OUT=R/'nb21_final_exact_n07_ensemble_fer';OUT.mkdir(exist_ok=True)
MODELS={'trpakov/vit-face-expression':'ef0bc6fc34241b6587e7e009e7711357be28c024',
'mo-thecreator/vit-Facial-Expression-Recognition':'5aae4f09e1da247eb0f1c1e6f53271601fcafe09'}
def sha(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for chunk in iter(lambda:f.read(1048576),b''):h.update(chunk)
 return h.hexdigest()
resolved=json.loads((N07/'resolved_config.json').read_text())
assert all(resolved['evaluators']['emotion_model_revisions'][k]==v for k,v in MODELS.items())
raw=pd.read_csv(N07/'raw_results.csv')
rec=pd.read_csv(N17/'private_execution_receipts.csv')
ref=pd.read_csv(N18/'private_paired_reference_and_n17_manifest.csv')
biom=pd.read_csv(N19/'private_v3_biometric_rows.csv')
assert len(raw)==600 and len(rec)==len(ref)==len(biom)==24
assert rec.source_id.nunique()==12 and set(rec.seed)=={42,123}
assert not rec.duplicated(['source_id','seed']).any()
assert (rec.status=='VERIFIED').all()
assert set(rec.runner_commit)=={'85594626161caca9dafa68592c1ffa21db87a806'}
assert set(rec.sd3_revision)=={'ea42f8cef0f178587cf766dc8129abd379c90671'}
for x in rec.itertuples(index=False):
 assert Path(x.output_path).is_file() and sha(x.output_path)==x.output_sha256
source=ref[['source_id','original_fer_predicted_class','human_target_emotion']].drop_duplicates()
assert len(source)==12 and not source.duplicated('source_id').any()
assert ref[['source_id','seed']].merge(rec[['source_id','seed']],validate='one_to_one').shape[0]==24
assert biom[['source_id','seed']].merge(rec[['source_id','seed']],validate='one_to_one').shape[0]==24
print('PREFLIGHT PASS: 12 independent sources; 24 verified N17 images; N07 two-ViT revisions pinned')

In [ ]:
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
E={}
for ident,revision in MODELS.items():
 processor=AutoImageProcessor.from_pretrained(ident,revision=revision)
 model=AutoModelForImageClassification.from_pretrained(ident,revision=revision).to(DEVICE)
 model.eval()
 E[ident]=(processor,model)
 print('Loaded:',ident,revision)

# Match N07 cell 12: model_emotion_distribution + ensemble_emotion_result.
# Explicitly use N07's own canonical aliases; no post-hoc mapping to improve matching.
CANON={'anger':'angry','happiness':'happy','sadness':'sad','surprised':'surprise',
       'fearful':'fear','neutrality':'neutral','disgusted':'disgust'}
def canonical(v):
 v=str(v).strip().lower()
 return CANON.get(v,v)

def evaluate(path):
 with Image.open(path) as handle: img=handle.convert('RGB')
 model_dists={}
 for ident,(processor,model) in E.items():
  inputs=processor(images=img,return_tensors='pt').to(DEVICE)
  with torch.inference_mode(): probs=torch.softmax(model(**inputs).logits,dim=-1)[0]
  d={}
  for j,p in enumerate(probs):
   label=canonical(model.config.id2label[j]);d[label]=d.get(label,0.0)+float(p.item())
  model_dists[ident]=d
 labels=sorted(set().union(*(set(v) for v in model_dists.values())))
 ensemble={k:float(np.mean([d.get(k,0.0) for d in model_dists.values()])) for k in labels}
 assert abs(sum(ensemble.values())-1)<1e-4
 return max(ensemble,key=ensemble.get),ensemble

rows=[]
for x in source.itertuples(index=False):
 p=N07/'sources'/f'{x.source_id}.png'
 assert p.is_file()
 prediction,_=evaluate(p)
 rows.append({'source_id':x.source_id,'expected_n07_ensemble':canonical(x.original_fer_predicted_class),
              'replayed_ensemble':prediction,'equal':prediction==canonical(x.original_fer_predicted_class)})
original_replay=pd.DataFrame(rows)
original_replay.to_csv(OUT/'private_original_ensemble_replay.csv',index=False)
count=int(original_replay.equal.sum())
print('N07 ENSEMBLE ORIGINAL REPLAY:',count,'/12')
print(original_replay.to_string(index=False))
if count!=12:
 status={'status':'BLOCKED_ENSEMBLE_ORIGINAL_PREDICTION_MISMATCH','match_count':count,'sources':12}
 (OUT/'evidence_status.json').write_text(json.dumps(status,indent=2))
 raise RuntimeError('STOP: N07 original predictions mismatch; inspect N07 EMOTION_CANON and source preprocessing before N17 scoring')

In [ ]:
cache=OUT/'private_n17_ensemble_predictions.csv'
prior=pd.read_csv(cache) if cache.exists() else pd.DataFrame()
if len(prior):
 assert not prior.duplicated(['source_id','seed']).any()
records=[]
for k,x in enumerate(rec.sort_values(['source_id','seed']).itertuples(index=False),1):
 hit=prior[(prior.source_id==x.source_id)&(prior.seed==x.seed)&(prior.output_sha256==x.output_sha256)] if len(prior) else pd.DataFrame()
 if len(hit)==1:
  records.append(hit.iloc[0].to_dict())
 else:
  prediction,distribution=evaluate(x.output_path)
  records.append({'source_id':x.source_id,'seed':int(x.seed),'output_sha256':x.output_sha256,
                  'n17_emotion_predicted_class':prediction,'target_model_probabilities_json':json.dumps(distribution,sort_keys=True),
                  'fer_evaluator_provenance':'N07_TWO_VIT_FROZEN_ENSEMBLE_REPLAY_12_OF_12'})
  pd.DataFrame(records).to_csv(cache,index=False)
 print(f'FER {k}/24',flush=True)
new=pd.DataFrame(records)
assert len(new)==24 and not new.duplicated(['source_id','seed']).any()
assert (new.fer_evaluator_provenance=='N07_TWO_VIT_FROZEN_ENSEMBLE_REPLAY_12_OF_12').all()
merged=ref.merge(new,on=['source_id','seed','output_sha256'],validate='one_to_one')
assert len(merged)==24
# N07 original definition: GENERATED predicted class == HUMAN TARGET.
merged['n17_emotion_preserved']=merged.n17_emotion_predicted_class.map(canonical).eq(merged.human_target_emotion.map(canonical))
# N07 baseline already contains exactly this definition, do not overwrite/reinterpret it.
merged['delta_fer_preserved']=merged.n17_emotion_preserved.astype(int)-merged.n07_emotion_preserved.astype(int)
merged.to_csv(OUT/'private_final_fer_paired.csv',index=False)
print('N07 human-target preservation:',int(merged.n07_emotion_preserved.sum()),'/24')
print('N17 human-target preservation:',int(merged.n17_emotion_preserved.sum()),'/24')

In [ ]:
# Fixed source-cluster CI; do not treat two seeds from the same face as independent.
def cluster_ci(frame,column,seed=21021,B=10000):
 rng=np.random.default_rng(seed)
 groups=[g[column].astype(float).to_numpy() for _,g in frame.groupby('source_id',sort=True)]
 samples=np.empty(B)
 for i in range(B):
  idx=rng.integers(0,len(groups),len(groups))
  samples[i]=np.concatenate([groups[j] for j in idx]).mean()
 return list(map(float,np.quantile(samples,[.025,.975])))
biom_metrics=pd.read_csv(N19/'private_paired_biometrics_and_niqe.csv')
assert len(biom_metrics)==24
joined=merged.merge(biom_metrics[['source_id','seed','delta_niqe','delta_arcface','delta_auraface']],
                    on=['source_id','seed'],validate='one_to_one')
assert len(joined)==24
metrics=['delta_fer_preserved','delta_arcface','delta_auraface','delta_niqe']
summary=[]
for metric in metrics:
 assert joined[metric].notna().all(),f'Missing measurements in {metric}'
 low,high=cluster_ci(joined,metric)
 summary.append({'metric':metric,'n_pairs':24,'n_sources':12,'mean_delta':float(joined[metric].mean()),
                 'ci95_low':low,'ci95_high':high,'bootstrap_draws':10000,'bootstrap_seed':21021})
sumdf=pd.DataFrame(summary)
sumdf.to_csv(OUT/'private_final_multimetric_summary.csv',index=False)
joined.to_csv(OUT/'private_full_multimetric_pairs.csv',index=False)
report={'status':'ENSEMBLE_REPLAY_AND_PAIRED_EVALUATION_COMPLETE_PENDING_MANUAL_QC',
        'n07_ensemble_original_replay_matches':12,'n17_predictions':24,
        'emotion_definition':'generated ensemble prediction equals human target label',
        'fer_models':MODELS,'independent_source_clusters':12,
        'manual_source_face_qc':'PENDING','calibrated_reidentification_risk':False}
(OUT/'evidence_status.json').write_text(json.dumps(report,indent=2))
print(sumdf.round(5).to_string(index=False))
print('FINISHED — private results:',OUT)
print('Share only aggregated summary, not private paired rows.')